In [ ]:
#Question 2.1
import numpy as np
from scipy import integrate
from scipy.stats import norm

mu = 0.5
sigma = 2.0
X =0.5

def binomiallogit(beta, X, pdf=None):

  logit = 1.0/(1.0+np.exp(-beta*X))
  if pdf is not None:
    return logit*pdf(beta)
  else:
    return logit * norm.pdf(beta, loc=mu, scale=sigma)

In [ ]:
#Question 2.2
true_value, true_error = integrate.quad(binomiallogit, -np.inf, np.inf, args=(X,), epsabs = 1e-14, epsrel=1e-14)
print(f"\nTrue value (quad, tol=1e-14): {true_value:.15f}")



True value (quad, tol=1e-14): 0.551493272636643


/var/folders/f_/hv79f9qs6rv1cnyrsbc7tydc0000gn/T/ipykernel_61264/847842575.py:14: RuntimeWarning: overflow encountered in exp
  logit = 1.0/(1.0+np.exp(-beta*X))


In [45]:
#Question 2.3
results_1d= []

np.random.seed(42)
for S in [20,400]:
  draws = norm.rvs(loc=mu, scale=sigma, size=S)
  logit_values = 1.0 / (1.0+np.exp(-draws*X))
  mc_est = np.mean(logit_values)
  print(f" S={S:>4d} Estimate = {mc_est:.12f}")
  
  results_1d.append({
        "Method": "Monte Carlo",
        "Setting": f"S={S}",
        "Points Used": S,
        "Estimate": mc_est,
        "Abs Error": abs(mc_est - true_value)
    })

 S=  20 Estimate = 0.516307014395
 S= 400 Estimate = 0.557198567335


In [ ]:
#Quetion 2.4
for k in [4,5,9,12]:
  nodes, weights = np.polynomial.hermite.hermgauss(k)
  beta_nodes = np.sqrt(2)*sigma*nodes+mu
  adj_weights = weights/np.sqrt(np.pi)

  logit_values = 1.0 / (1.0+np.exp(-beta_nodes*X))
  gh_est = np.sum(adj_weights*logit_values)
  print(f"k={k:>2d} Estimate = {gh_est:.12f}")

  results_1d.append({
        "Method": "Gauss-Hermite",
        "Setting": f"k={k}",
        "Points Used": k,
        "Estimate": gh_est,
        "Abs Error": abs(gh_est - true_value)
    })

k= 4 Estimate = 0.551313876771
k= 5 Estimate = 0.551549103736
k= 9 Estimate = 0.551494224291
k=12 Estimate = 0.551493204065


In [53]:
#Question 2.5
print("The Guass_Hermite quadrature estimates are much more accurate than the Monte Carlo estimates. Even the lower order quadratures preform better than the Monte Carlo estimate with 20 draws and even better than " \
"the one with 400 draws. ")

The Guass_Hermite quadrature estimates are much more accurate than the Monte Carlo estimates. Even the lower order quadratures preform better than the Monte Carlo estimate with 20 draws and even better than the one with 400 draws. 


In [51]:
#Question 2.6
mu2 = np.array([0.5, 1.0])
sigma2 = np.array([2.0,1.0])
X2 = np.array([0.5, 1.0])

def binomiallogit_2d(beta1, beta2):
  v =beta1*X2[0]+beta2*X2[1]
  logit = 1.0/(1.0+np.exp(-v))
  return logit * norm.pdf(beta1, mu2[0], sigma2[0]) * norm.pdf(beta2, mu2[1], sigma2[1])

true_2d, err_2d = integrate.dblquad(binomiallogit_2d,-np.inf, np.inf, lambda x: -np.inf,lambda x: np.inf, epsabs=1e-14, epsrel=1e-14)

results_2d = []

np.random.seed(42)
for S in [20,400]:
  d1 = norm.rvs(loc=mu2[0], scale=sigma2[0], size=S)
  d2 = norm.rvs(loc=mu2[1], scale=sigma2[1], size=S)
  v = d1*X2[0]+d2*X2[1]
  logit_values = 1.0 / (1.0+np.exp(-v))
  mc_est = np.mean(logit_values)
  print(f" S={S:>4d} Estimate = {mc_est:.12f}")

  results_2d.append({
        "Method": "Monte Carlo",
        "Setting": f"S={S}",
        "Points Used": S,
        "Estimate": mc_est,
        "Abs Error": abs(mc_est - true_2d)
    })


for k in [4,5,9,12]:
  nodes, weights = np.polynomial.hermite.hermgauss(k)
  b1 = np.sqrt(2)*sigma2[0]*nodes+mu2[0]
  b2 = np.sqrt(2)*sigma2[1]*nodes+mu2[1]
  w = weights/np.sqrt(np.pi)

  est = 0.0
  for i in range(k):
    for j in range(k):
      v = b1[i]*X2[0]+b2[j]*X2[1]
      est += w[i]*w[j]/(1.0 + np.exp(-v))

  print(f"estimate = {est: .12f}")

  results_2d.append({
        "Method": "Gauss-Hermite",
        "Setting": f"k={k}",
        "Points Used": k**2,
        "Estimate": est,
        "Abs Error": abs(est - true_2d)
    })


/var/folders/f_/hv79f9qs6rv1cnyrsbc7tydc0000gn/T/ipykernel_61264/2798658809.py:8: RuntimeWarning: overflow encountered in exp
  logit = 1.0/(1.0+np.exp(-v))


 S=  20 Estimate = 0.651319192494
 S= 400 Estimate = 0.721990738592
estimate =  0.714371378972
estimate =  0.714485614767
estimate =  0.714483763460
estimate =  0.714483809858


In [52]:
#Question 2.7
table_1d = pd.DataFrame(results_1d)
print("1D Table:")
print(table_1d.round(12))

table_2d = pd.DataFrame(results_2d)
print("2D Table:")
print(table_2d.round(12))

1D Table:
          Method Setting  Points Used  Estimate     Abs Error
0    Monte Carlo    S=20           20  0.516307  3.518626e-02
1    Monte Carlo   S=400          400  0.557199  5.705295e-03
2  Gauss-Hermite     k=4            4  0.551314  1.793959e-04
3  Gauss-Hermite     k=5            5  0.551549  5.583110e-05
4  Gauss-Hermite     k=9            9  0.551494  9.516540e-07
5  Gauss-Hermite    k=12           12  0.551493  6.857100e-08
2D Table:
          Method Setting  Points Used  Estimate     Abs Error
0    Monte Carlo    S=20           20  0.651319  6.316461e-02
1    Monte Carlo   S=400          400  0.721991  7.506933e-03
2  Gauss-Hermite     k=4           16  0.714371  1.124264e-04
3  Gauss-Hermite     k=5           25  0.714486  1.809372e-06
4  Gauss-Hermite     k=9           81  0.714484  4.193400e-08
5  Gauss-Hermite    k=12          144  0.714484  4.464000e-09


In [34]:
##Question 3.1
import numpy as np
import pandas as pd
from scipy.optimize import minimize

df = pd.read_csv("pset1_data.csv").sort_values(["individual_id_i", "apt_id_j"]).reset_index(drop=True)

N = df["individual_id_i"].nunique()
J = df.groupby("individual_id_i").size().iloc[0]
y = df["y_ij"].to_numpy().reshape(N, J)
logsq = df["logsqfeet_j"].to_numpy().reshape(N, J)
bath = df["bath_j"].to_numpy().reshape(N, J)
fam = df["family_size_i"].to_numpy().reshape(N, J)
outdoor = df["outdoor_j"].to_numpy().reshape(N, J)
bathfam = bath * fam

def neg_loglike(parameters):
    beta0,beta1,beta2,beta3,gamma = parameters
    V = beta0 + beta1 * logsq+ beta2 * bath + beta3 * bathfam + gamma * outdoor
    row_max = np.maximum(0.0, V.max(axis=1, keepdims=True))
    expV = np.exp(V-row_max)

    denom = np.exp(-row_max)+expV.sum(axis=1, keepdims=True)
    P = expV/denom

    P0 = np.exp(-row_max[:,0]) / denom[:,0]
    P = np.clip(P,1e-300, 1.0)
    P0 = np.clip(P0, 1e-300, 1.0)

    LL = np.sum(y*np.log(P)) + np.sum((1-y.sum(axis=1)) *np.log(P0))
    return -LL


def grad(parameters): 
    beta0,beta1,beta2,beta3,gamma = parameters
    V = beta0 + beta1 * logsq+ beta2 * bath + beta3 * bathfam + gamma * outdoor
    row_max = np.maximum(0.0, V.max(axis=1, keepdims=True))
    expV = np.exp(V-row_max)
    denom = np.exp(-row_max)+expV.sum(axis=1, keepdims=True)
    P = expV/denom

    resid = y-P

    g0 = resid.sum()
    g1 = (resid*logsq).sum()
    g2 = (resid*bath).sum()
    g3 = (resid*bathfam).sum()
    g4 = (resid*outdoor).sum()

    return -np.array([g0,g1,g2,g3,g4])

x0 = np.zeros(5)
res = minimize(neg_loglike, x0, jac=grad, method="L-BFGS-B")

print("Estimates:")
print(f"beta0  = {res.x[0]:.6f}")
print(f"beta1  = {res.x[1]:.6f}")
print(f"beta2  = {res.x[2]:.6f}")
print(f"beta3  = {res.x[3]:.6f}")
print(f"gamma  = {res.x[4]:.6f}")


Estimates:
beta0  = -4.813499
beta1  = -0.109351
beta2  = 0.767751
beta3  = 0.042052
gamma  = 0.809408


In [44]:
#Question 3.2
from numpy.polynomial.hermite import hermgauss

S= 12
nodes, weights = hermgauss(S)
weights = weights / np.sqrt(np.pi)

def neg_loglike_mixed(paramaters):
    beta0,beta1,beta2,beta3,gamma,sigma = paramaters
    sigma = np.abs(sigma)

    P_avg = np.zeros((N,J))
    P0_avg = np.zeros(N)

    for s in range(S):
        gamma_s = gamma + np.sqrt(2)*sigma*nodes[s]
        V = beta0 + beta1 * logsq+ beta2 * bath + beta3 * bathfam + gamma_s * outdoor

        row_max = np.maximum(0.0, V.max(axis=1, keepdims=True))
        expV = np.exp(V-row_max)
        denom = np.exp(-row_max)+expV.sum(axis=1, keepdims=True)
        P = expV/denom
        P0 = np.exp(-row_max[:, 0])/denom[:,0]
        P_avg += weights[s]*P
        P0_avg += weights[s]*P0
    
    P_avg = np.clip(P_avg, 1e-300, 1.0)
    P0_avg = np.clip(P0_avg, 1e-300, 1.0)

    LL = np.sum(y*np.log(P_avg)) + np.sum((1-y.sum(axis=1)) *np.log(P0_avg))

    return -LL


x0_mixed = np.append(res.x, 0.5)
res_mixed = minimize(neg_loglike_mixed, x0_mixed, method="L-BFGS-B")

print("Mixed Logit Estimates:")
print(f"beta0  = {res_mixed.x[0]:.6f}")
print(f"beta1  = {res_mixed.x[1]:.6f}")
print(f"beta2  = {res_mixed.x[2]:.6f}")
print(f"beta3  = {res_mixed.x[3]:.6f}")
print(f"gamma  = {res_mixed.x[4]:.6f}")
print(f"sigma  = {abs(res_mixed.x[5]):.6f}")

Mixed Logit Estimates:
beta0  = -4.300228
beta1  = 0.468248
beta2  = 0.328722
beta3  = 0.071593
gamma  = 1.071018
sigma  = 2.148300
